In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

test.head()


/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv
Path to competition files: /kaggle/input/competitions/titanic


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [2]:
# percentage of women who has survived
total_women = train.loc[train.Sex=='female']
Survived_women = total_women.loc[(total_women.Survived == 1)]

Percentage = len(Survived_women)/len(total_women)
print(f"Women Perentage : {Percentage:.2%}")

# percentage of men who has survived
total_men = train.loc[train.Sex=='male']
Survived_men = total_men.loc[(total_men.Survived == 1)]

Percentage = len(Survived_men)/len(total_men)
print(f"men Perentage : {Percentage:.2%}")

Women Perentage : 74.20%
men Perentage : 18.89%


In [3]:
# Feature Engineering

# Create a list containing both datasets

dfs = [train, test]

for df in dfs:
    # 1. Extract Title
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    df["Title"] = df["Title"].replace(
        [
            "Lady",
            "Countess",
            "Capt",
            "Col",
            "Don",
            "Dr",
            "Major",
            "Rev",
            "Sir",
            "Jonkheer",
            "Dona",
        ],
        "Rare",
    )
    df["Title"] = df["Title"].replace(
        {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
    )

    # 2. Family Features
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    df['Fare'] = df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
    df['Age'] = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))


    


In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 5, 8],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

y = train["Survived"]

features = ["Pclass", "Sex","Embarked","Title","FamilySize","IsAlone", "Age","Fare"]
X = pd.get_dummies(train[features])
X_test = pd.get_dummies(test[features])

X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)


rf = RandomForestClassifier(random_state=1)
R_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

R_search.fit(X, y)

print("Best Parameters:", R_search.best_params_)
print("Best Cross-Validation Score:", R_search.best_score_)

# Overwrite your model with the optimized version
model = R_search.best_estimator_



Best Parameters: {'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 200}
Best Cross-Validation Score: 0.8338710689849979


In [5]:
# Run K-Fold Cross-Validation
from sklearn.model_selection import cross_validate

cv_results = cross_validate(
    model, X, y, cv=5, scoring="accuracy", return_train_score=True
)

# Calculate averages
train_score = cv_results["train_score"].mean()
cv_score = cv_results["test_score"].mean()

print(f"Train Accuracy: {train_score:.4f}")
print(f"CV Accuracy:    {cv_score:.4f}")
print(f"Gap (Overfit):  {train_score - cv_score:.4f}")


Train Accuracy: 0.8432
CV Accuracy:    0.8339
Gap (Overfit):  0.0093


In [6]:
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
